In [ ]:
!nvidia-smi

Fri May  1 16:08:44 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

======== Error: no application specified.


In [1]:
!pip install py-cpuinfo pynvml gputil psutil

  Preparing metadata (setup.py) ... done
  Created wheel for gputil: filename=GPUtil-1.4.0-py3-none-any.whl size=7392 sha256=623a8a18d8b2c6120fe0fb290134e1ec9e89a236f1478ff12f2c4a65f5fc4ec9
  Stored in directory: /root/.cache/pip/wheels/92/a8/b7/d8a067c31a74de9ca252bbe53dea5f896faabd25d55f541037
Successfully built gputil


In [3]:
import sys
import subprocess
import psutil
import platform
import os
import shutil
from cpuinfo import get_cpu_info
import torch
import GPUtil
try:
    import pynvml
    pynvml_available = True
except ImportError:
    pynvml_available = False

def run_cmd(cmd):
    try:
        return subprocess.check_output(cmd, shell=True, text=True).strip()
    except:
        return "N/A"

def get_gpu_details_nvml():
    """Детальная информация о GPU через NVML"""
    if not pynvml_available:
        return {}
    pynvml.nvmlInit()
    device_count = pynvml.nvmlDeviceGetCount()
    if device_count == 0:
        return {}
    handle = pynvml.nvmlDeviceGetHandleByIndex(0)
    info = {}
    try:
        info['name'] = pynvml.nvmlDeviceGetName(handle).decode()
    except:
        info['name'] = "N/A"
    try:
        cc = pynvml.nvmlDeviceGetCudaComputeCapability(handle)
        info['compute_capability'] = f"{cc[0]}.{cc[1]}"
    except:
        info['compute_capability'] = "N/A"
    try:
        mem_info = pynvml.nvmlDeviceGetMemoryInfo(handle)
        info['video_memory_mb'] = mem_info.total // (1024**2)
    except:
        info['video_memory_mb'] = "N/A"
    try:
        bus_width_str = run_cmd("nvidia-smi --query-gpu=memory.bus_width --format=csv,noheader")
        info['memory_interface_bits'] = bus_width_str if bus_width_str and "N/A" not in bus_width_str else "N/A"
    except:
        info['memory_interface_bits'] = "N/A"
    try:
        sm_str = run_cmd("nvidia-smi --query-gpu=sm_count --format=csv,noheader")
        info['sm_count'] = sm_str if sm_str and "N/A" not in sm_str else "N/A"
    except:
        info['sm_count'] = "N/A"
    return info

def get_cpu_details_full():
    info = get_cpu_info()
    model = info.get('brand_raw', 'N/A')
    physical_cores = psutil.cpu_count(logical=False)
    logical_cores = psutil.cpu_count(logical=True)
    freq = psutil.cpu_freq()
    base_freq = freq.min if freq else 0.0
    max_freq = freq.max if freq else 0.0
    l1_data = info.get('l1_data_cache_size', 'N/A')
    l1_inst = info.get('l1_instruction_cache_size', 'N/A')
    l2_cache = info.get('l2_cache_size', 'N/A')
    l3_cache = info.get('l3_cache_size', 'N/A')
    flags = info.get('flags', [])
    vmx = 'VT-x' in flags or 'vmx' in flags
    svm = 'SVM' in flags or 'svm' in flags
    virt = "Enabled (VT-x)" if vmx else ("Enabled (SVM)" if svm else "N/A")
    # Пытаемся определить кол-во P-cores и E-cores (только для Intel Alder Lake+)
    # В Colab обычно нет E-cores, поэтому просто выводим общее число
    return {
        'model': model,
        'total_cores': f"{physical_cores} (все логические)" if physical_cores else "N/A",
        'total_threads': logical_cores,
        'base_freq_ghz': base_freq / 1000.0 if base_freq else 'N/A',
        'max_freq_ghz': max_freq / 1000.0 if max_freq else 'N/A',
        'l3_cache_mb': l3_cache,
        'l2_cache_mb': l2_cache,
        'l1_cache_kb': l1_data,
        'virt_enabled': virt
    }

print("="*60)
print("СБОР ХАРАКТЕРИСТИК СИСТЕМЫ")
print("="*60)

# 1. GPU
print("\n--- ГРАФИЧЕСКИЙ ПРОЦЕССОР ---")
gpu_nvml = get_gpu_details_nvml()
if gpu_nvml:
    print(f"Модель: {gpu_nvml.get('name', 'N/A')}")
    print(f"Compute Capability: {gpu_nvml.get('compute_capability', 'N/A')}")
    print(f"Видеопамять: {gpu_nvml.get('video_memory_mb', 'N/A')} МБ")
    print(f"Шина памяти: {gpu_nvml.get('memory_interface_bits', 'N/A')}-бит")
    print(f"Количество SM: {gpu_nvml.get('sm_count', 'N/A')}")

    # Расчёт CUDA/Tensor/RT ядер по SM (только для архитектур Turing/Ampere/Ada)
    sm = gpu_nvml.get('sm_count')
    if sm != 'N/A' and sm.isdigit():
        sm = int(sm)
        # Для T4 (Turing): 64 CUDA на SM, 8 Tensor на SM, RT cores = 0
        cuda_per_sm = 64
        tensor_per_sm = 8
        rt_per_sm = 0
        print(f"Количество ядер CUDA: {sm * cuda_per_sm}")
        print(f"Количество тензорных ядер: {sm * tensor_per_sm}")
        print(f"Количество RT-ядер: {sm * rt_per_sm}")
    else:
        print("Не удалось вычислить количество ядер (нет данных о SM)")

    # Детали через PyTorch (shared memory, max threads, blocks)
    if torch.cuda.is_available():
        prop = torch.cuda.get_device_properties(0)
        print(f"Разделяемая память на блок (Shared Memory): {prop.shared_memory_per_block // 1024} КБ")
        print(f"Макс. количество нитей на блок (Max Threads): {prop.max_threads_per_block}")
        # Макс. блоков: в PyTorch нет прямого поля, но обычно 2^31-1
        print(f"Макс. количество блоков (грид): 2³¹-1 (не ограничено в рамках памяти)")
        # Константная память (Constant Memory): через CUDA Runtime API программно не достать, но для T4 = 64 КБ
        print(f"Константная память (Constant Memory): 64 КБ (справочное значение для Turing/Ampere)")
        # Регистров на блок – не доступно в PyTorch, используем nvidia-smi или справочник
        print(f"Регистров на блок: не детектируется программно. Для T4: 65536 регистров на блок (макс. 255 на поток)")
    else:
        print("CUDA не доступна, детали через PyTorch не получить")
else:
    print("pynvml не удалось получить данные. Использую GPUtil:")
    gpus = GPUtil.getGPUs()
    if gpus:
        g = gpus[0]
        print(f"Модель: {g.name}")
        print(f"Видеопамять: {g.memoryTotal} МБ")
        print(f"Compute Capability: {g.computeCapability if hasattr(g, 'computeCapability') else 'N/A'}")

# 2. CPU
print("\n--- ПРОЦЕССОР ---")
cpu = get_cpu_details_full()
print(f"Модель: {cpu['model']}")
print(f"Всего ядер: {cpu['total_cores']}")
print(f"Всего потоков: {cpu['total_threads']}")
print(f"Базовая частота: {cpu['base_freq_ghz']} ГГц")
print(f"Макс. частота Turbo: {cpu['max_freq_ghz']} ГГц")
print(f"Кэш L3: {cpu['l3_cache_mb']}")
print(f"Кэш L2: {cpu['l2_cache_mb']}")
print(f"Кэш L1: {cpu['l1_cache_kb']}")
print(f"Виртуализация: {cpu['virt_enabled']}")

# 3. Оперативная память
print("\n--- ОПЕРАТИВНАЯ ПАМЯТЬ ---")
ram = psutil.virtual_memory()
print(f"Объем: {ram.total // (1024**3)} ГБ")

# 4. Диск
print("\n--- ДИСК ---")
disk = shutil.disk_usage("/")
print(f"Объем: {disk.total // (1024**3)} ГБ")

# 5. ОС, IDE, компилятор
print("\n--- ПРОГРАММНОЕ ОКРУЖЕНИЕ ---")
os_name = platform.system() + " " + platform.release()
ide = "N/A (Google Colab environment)"
nvcc_ver = run_cmd("nvcc --version | grep 'release' | awk '{print $6}' | tr -d ','")
compiler = f"NVIDIA CUDA Compiler ({nvcc_ver})" if nvcc_ver != "N/A" else "NVIDIA CUDA Compiler (unknown version)"
print(f"OS: {os_name}")
print(f"IDE: {ide}")
print(f"Compiler: {compiler}")

print("\n" + "="*60)
print("Примечание: регистры на блок и константная память не извлекаются стандартными Python-библиотеками.")
print("Для T4 эталонные значения: Регистров на блок = 65536, Константная память = 64 KB, Макс. блоков = 2³¹-1.")
print("Если необходимы точные значения из драйвера, используйте pycuda (см. дополнительный блок ниже).")
print("="*60)

СБОР ХАРАКТЕРИСТИК СИСТЕМЫ

--- ГРАФИЧЕСКИЙ ПРОЦЕССОР ---
Модель: N/A
Compute Capability: 7.5
Видеопамять: 15360 МБ
Шина памяти: N/A-бит
Количество SM: N/A
Не удалось вычислить количество ядер (нет данных о SM)
Разделяемая память на блок (Shared Memory): 48 КБ
Макс. количество нитей на блок (Max Threads): 1024
Макс. количество блоков (грид): 2³¹-1 (не ограничено в рамках памяти)
Константная память (Constant Memory): 64 КБ (справочное значение для Turing/Ampere)
Регистров на блок: не детектируется программно. Для T4: 65536 регистров на блок (макс. 255 на поток)

--- ПРОЦЕССОР ---
Модель: Intel(R) Xeon(R) CPU @ 2.00GHz
Всего ядер: 1 (все логические)
Всего потоков: 2
Базовая частота: N/A ГГц
Макс. частота Turbo: N/A ГГц
Кэш L3: 40370176
Кэш L2: 1048576
Кэш L1: 32768
Виртуализация: N/A

--- ОПЕРАТИВНАЯ ПАМЯТЬ ---
Объем: 12 ГБ

--- ДИСК ---
Объем: 112 ГБ

--- ПРОГРАММНОЕ ОКРУЖЕНИЕ ---
OS: Linux 6.6.122+
IDE: N/A (Google Colab environment)
Compiler: NVIDIA CUDA Compiler (V12.8.93)

Примечани

In [4]:
!pip install pycuda

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 32.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.1/99.1 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.2/103.2 kB 13.9 MB/s eta 0:00:00
  Created wheel for pycuda: filename=pycuda-2026.1-cp312-cp312-linux_x86_64.whl size=659447 sha256=89ed73acbbe75fb72c623ed4e00c019990deae5d69e71a1d6b5b61d05920da01
  Stored in directory: /root/.cache/pip/wheels/90/2a/71/75ec0cc316cc0ff494bfffa2935e02580129cb7f859a0cfd8f
Successfully built pycuda


In [6]:
import pycuda.driver as cuda
import pycuda.autoinit

device = cuda.Device(0)
props = device.get_attributes()

# Словарь атрибутов:
# cuda.device_attribute.REGISTERS_PER_BLOCK -> 255? нет, это регистры на поток.
# Правильное название: REGISTERS_PER_BLOCK = 255? На самом деле это максимальное количество регистров на блок (в единицах по 32 бита)
# Для T4: значение 65536 (или 64K). Но в pycuda атрибут называется REGISTERS_PER_BLOCK.
# Константная память: TOTAL_CONSTANT_MEMORY

regs_per_block = props[cuda.device_attribute.REGISTERS_PER_BLOCK]
const_mem = props[cuda.device_attribute.TOTAL_CONSTANT_MEMORY]

print(f"Регистров на блок (max): {regs_per_block}")
print(f"Константная память (байт): {const_mem} ({const_mem//1024} КБ)")

Регистров на блок (max): 65536
Константная память (байт): 65536 (64 КБ)


In [4]:
%%writefile kernel.cu
#include <cstdio>
#include <cstdlib>
#include <cuda_runtime.h>
#include <cuda/pipeline>

static constexpr int BLOCK = 256;
static constexpr int LOCAL = 1024;
static constexpr int INF = 0x7fffffff;

__device__ __forceinline__ void swapDir(int &a, int &b, int dir) {
    int mn = min(a, b);
    int mx = max(a, b);
    a = dir ? mn : mx;
    b = dir ? mx : mn;
}

__global__ void oddEvenShared(int *data, int n) {
    __shared__ int s[LOCAL + 1];

    int tid = threadIdx.x;
    int base = blockIdx.x << 10;
    int gid = base + tid;

#if __CUDA_ARCH__ >= 800
    cuda::pipeline<cuda::thread_scope_thread> pipe = cuda::make_pipeline();
    if (gid < n)
        cuda::memcpy_async(&s[tid], &data[gid], sizeof(int), pipe);
    else
        s[tid] = INF;
    pipe.commit();
    pipe.wait_prior<0>();
#else
    s[tid] = (gid < n) ? data[gid] : INF;
#endif

    __syncthreads();

    int lim = n - base;
    if (lim > LOCAL) lim = LOCAL;

    for (int phase = 0; phase < lim; ++phase) {
        int idx = (tid << 1) + (phase & 1);
        if (idx + 1 < lim) {
            int a = s[idx];
            int b = s[idx + 1];
            int mn = min(a, b);
            int mx = max(a, b);
            s[idx] = mn;
            s[idx + 1] = mx;
        }
        __syncthreads();
    }

    if (blockIdx.x & 1) {
        int rid = lim - tid - 1;
        if (tid < (lim >> 1)) {
            int tmp = s[tid];
            s[tid] = s[rid];
            s[rid] = tmp;
        }
        __syncthreads();
    }

    if (gid < n)
        data[gid] = s[tid];
}

__global__ void bitonicShared(int *data, int n, int j, int k) {
    __shared__ int s[LOCAL + 1];

    int tid = threadIdx.x;
    int base = blockIdx.x << 10;
    int gid = base + tid;

#if __CUDA_ARCH__ >= 800
    cuda::pipeline<cuda::thread_scope_thread> pipe = cuda::make_pipeline();
    if (gid < n)
        cuda::memcpy_async(&s[tid], &data[gid], sizeof(int), pipe);
    else
        s[tid] = INF;
    pipe.commit();
    pipe.wait_prior<0>();
#else
    s[tid] = (gid < n) ? data[gid] : INF;
#endif

    __syncthreads();

    if (tid < k) {
        int ixj = tid ^ j;
        if (ixj < k && ixj > tid) {
            int dir = ((gid & k) == 0);
            int a = s[tid];
            int b = s[ixj];
            swapDir(a, b, dir);
            s[tid] = a;
            s[ixj] = b;
        }
    }

    __syncthreads();

    if (gid < n)
        data[gid] = s[tid];
}

__global__ void bitonicGlobalShared(int *data, int n, int j, int k) {
    __shared__ int s[(BLOCK << 1) + 1];

    int tid = threadIdx.x;
    int base = blockIdx.x << 9;
    int gid = base + tid;

    if (gid < n)
        s[tid] = data[gid];
    else
        s[tid] = INF;

    __syncthreads();

    if (tid < (BLOCK << 1)) {
        int ixj = tid ^ j;
        if (ixj < (BLOCK << 1) && ixj > tid) {
            int dir = ((gid & k) == 0);
            int a = s[tid];
            int b = s[ixj];
            swapDir(a, b, dir);
            s[tid] = a;
            s[ixj] = b;
        }
    }

    __syncthreads();

    if (gid < n)
        data[gid] = s[tid];
}

__global__ void bitonicGlobal(int *data, int n, int j, int k) {
    int gid = (blockIdx.x << 8) + threadIdx.x;
    int ixj = gid ^ j;

    if (ixj > gid && ixj < n) {
        int dir = ((gid & k) == 0);
        int a = data[gid];
        int b = data[ixj];
        swapDir(a, b, dir);
        data[gid] = a;
        data[ixj] = b;
    }
}

int main() {
    int n;
    fread(&n, sizeof(int), 1, stdin);

    size_t bytes_orig = (size_t)n * sizeof(int);
    int *h_orig = (int*)aligned_alloc(128, bytes_orig);
    fread(h_orig, sizeof(int), n, stdin);

    int n_pow2 = 1;
    while (n_pow2 < n) n_pow2 <<= 1;
    size_t bytes_pow2 = (size_t)n_pow2 * sizeof(int);

    int *h_padded = (int*)aligned_alloc(128, bytes_pow2);
    for (int i = 0; i < n; ++i)
        h_padded[i] = h_orig[i];
    for (int i = n; i < n_pow2; ++i)
        h_padded[i] = INF;

    int *d;
    cudaMalloc(&d, bytes_pow2);
    cudaMemcpy(d, h_padded, bytes_pow2, cudaMemcpyHostToDevice);

    cudaStream_t stream;
    cudaStreamCreate(&stream);

    int blocks_local = (n_pow2 + LOCAL - 1) >> 10;
    oddEvenShared<<<blocks_local, LOCAL, 0, stream>>>(d, n_pow2);
    cudaDeviceSynchronize();

    for (int k = LOCAL << 1; k <= n_pow2; k <<= 1) {
        for (int j = k >> 1; j > 0; j >>= 1) {
            if (k <= LOCAL) {
                int grid = (n_pow2 + LOCAL - 1) >> 10;
                bitonicShared<<<grid, LOCAL, 0, stream>>>(d, n_pow2, j, k);
            } else if (j <= BLOCK) {
                int grid = (n_pow2 + (BLOCK << 1) - 1) >> 9;
                bitonicGlobalShared<<<grid, BLOCK << 1, 0, stream>>>(d, n_pow2, j, k);
            } else {
                int grid = (n_pow2 + BLOCK - 1) >> 8;
                bitonicGlobal<<<grid, BLOCK, 0, stream>>>(d, n_pow2, j, k);
            }
        }
    }

    cudaDeviceSynchronize();
    cudaMemcpy(h_orig, d, bytes_orig, cudaMemcpyDeviceToHost);
    fwrite(h_orig, sizeof(int), n, stdout);

    cudaStreamDestroy(stream);
    cudaFree(d);
    free(h_orig);
    free(h_padded);
    return 0;
}

Overwriting kernel.cu


In [5]:
%%shell
nvcc -arch=sm_75 kernel.cu -o a.out


In [7]:
%%shell
./a.out < input.data > out.data

In [3]:
%%shell
ncu --metrics l1tex__t_sectors_pipe_lsu_mem_global_op_ld,l1tex__t_sectors_pipe_lsu_mem_global_op_st,smsp__branch_targets_threads_divergent,sm__sass_inst_executed_op_local,l1tex__data_bank_conflicts_pipe_lsu_mem_shared_op_ld,l1tex__data_bank_conflicts_pipe_lsu_mem_shared_op_st ./a.out < matrix_1024.txt > out.txt 2>&1

/bin/bash: line 1: matrix_1024.txt: No such file or directory


CalledProcessError: Command 'ncu --metrics l1tex__t_sectors_pipe_lsu_mem_global_op_ld,l1tex__t_sectors_pipe_lsu_mem_global_op_st,smsp__branch_targets_threads_divergent,sm__sass_inst_executed_op_local,l1tex__data_bank_conflicts_pipe_lsu_mem_shared_op_ld,l1tex__data_bank_conflicts_pipe_lsu_mem_shared_op_st ./a.out < matrix_1024.txt > out.txt 2>&1
' returned non-zero exit status 1.

In [34]:
%%shell
cat out.txt

Выходные данные были обрезаны до нескольких последних строк (5000).
    l1tex__data_bank_conflicts_pipe_lsu_mem_shared_op_st.sum                        0
    l1tex__t_sectors_pipe_lsu_mem_global_op_ld.avg                sector        80.92
    l1tex__t_sectors_pipe_lsu_mem_global_op_ld.max                sector          312
    l1tex__t_sectors_pipe_lsu_mem_global_op_ld.min                sector           28
    l1tex__t_sectors_pipe_lsu_mem_global_op_ld.sum                sector        3,237
    l1tex__t_sectors_pipe_lsu_mem_global_op_st.avg                sector        22.50
    l1tex__t_sectors_pipe_lsu_mem_global_op_st.max                sector          256
    l1tex__t_sectors_pipe_lsu_mem_global_op_st.min                sector            0
    l1tex__t_sectors_pipe_lsu_mem_global_op_st.sum                sector          900
    sm__sass_inst_executed_op_local.avg                             inst            0
    sm__sass_inst_executed_op_local.max                             inst